# CS 3110/5110: Data Privacy
## In-Class Exercise, week of 11/9/2026

In [1]:
# Load the data and libraries
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)

def gaussian_mech(v, sensitivity, epsilon, delta):
    return v + np.random.normal(loc=0, scale=sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon)

def gaussian_mech_vec(vec, sensitivity, epsilon, delta):
    return [v + np.random.normal(loc=0, scale=sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon)
            for v in vec]

def pct_error(orig, priv):
    return np.abs(orig - priv)/orig * 100.0

adult = pd.read_csv('https://github.com/jnear/cs3110-data-privacy/raw/main/homework/adult_with_pii.csv')

## Question 1

Implement an *encoding function* for randomized response to a "yes/no" question.

Reference [Chapter 13](https://uvm-plaid.github.io/programming-dp/notebooks/ch13.html#randomized-response).

In [19]:
def encode_rand_resp_yes_no(true_response):
    first_coin = np.random.randint(0, 2)

    if first_coin == 0:
        return true_response
    else:
        return np.random.randint(0, 2) == 0

for _ in range(5):
    print('Randomized response:', encode_rand_resp_yes_no(True))

Randomized response: True
Randomized response: True
Randomized response: True
Randomized response: True
Randomized response: True


In [5]:
# TEST CASE
assert np.sum([encode_rand_resp_yes_no(True) for _ in range(100)]) > 60
assert np.sum([encode_rand_resp_yes_no(True) for _ in range(100)]) < 90

## Question 2

Implement a *decoding function* for randomized response to a "yes/no" question.

In [37]:
def decode_rand_resp_yes_no(responses):
    # return np.sum(responses)
    # We expect 1/2 to answer randomly
    # So a total of 1/4 people will say yes by randomness alone
    fake_responses = len(responses)/2
    # Total # of yesses recored
    num_yesses = np.sum(responses)
    # of expected real yesses is total # of yesses * 2 - # of fake responses
    true_yesses = num_yesses * 2 - fake_responses
    return true_yesses

# Example: 1000 "yesses" and 500 "nos"
true_responses = [True for _ in range(1000)] + [False for _ in range(500)]
print('Number of "True" yesses:', np.sum(true_responses))

# Randomized responses
# Each response satisfies 1.09-differential privacy
rand_responses = [encode_rand_resp_yes_no(r) for r in true_responses]

# Decode the responses by subtracting "fake" yesses
print('Decoded randomized response yesses:', decode_rand_resp_yes_no(rand_responses))

Number of "True" yesses: 1000
Decoded randomized response yesses: 954.0


In [ ]:
# TEST CASE
true_responses = [True for _ in range(1000)] + [False for _ in range(500)]

# Randomized responses
# Each response satisfies 1.09-differential privacy
rand_responses = [encode_rand_resp_yes_no(r) for r in true_responses]

# Decode the responses by subtracting "fake" yesses
assert decode_rand_resp_yes_no(rand_responses) < 1100
assert decode_rand_resp_yes_no(rand_responses) > 900

## Question 3

Use the definition of randomized response above to answer the question:

*How many individuals in the `adult` dataset have `Occupation` = `Sales`?*

Could:

Ask each person if they are in sales, 0 or 1

add up 1's to get # of salesmen

Idea:

Each person returns laplace_mech(ans, sens=1, epsilon=epsilon), sum

Problem, this is super noisy, noise for each answer

In [48]:
# take a single occupation from the adult dataset, and return a single response
def encode_response_sales(response):
    return encode_rand_resp_yes_no(response == 'Sales')

    
def decode_responses_sales(responses):
    return decode_rand_resp_yes_no(responses)

responses = [encode_response_sales(r) for r in adult['Occupation']]
decode_responses_sales(responses)

np.float64(3623.5)

In [51]:
# How accurate is the answer above?
true_sales = np.sum(adult['Occupation'] == 'Sales')
print('True number of salespeople:', true_sales)
print(f"Laplace result {laplace_mech(true_sales, sensitivity=1, epsilon=np.log(3))}")

True number of salespeople: 3650
Laplace result 3649.8658477202616


## Question 4

Implement the *encode* and *perturb* steps for Optimized Unary Hashing.

In [56]:
domain = adult['Occupation'].dropna().unique()
domain

# Takes response, encode it in vector of 1s and 0s
def encode(response):
    return [1 if d == response else 0 for d in domain]

encode('Sales')

[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]

In [69]:
domain = adult['Occupation'].dropna().unique()
domain

# Takes response, encode it in vector of 1s and 0s
def encode(response):
    return [1 if d == response else 0 for d in domain]

encode('Sales')

# Randomizes above vecetor
def perturb(encoded_response):
    def perturb_bit(bit):
        # Adjusts epsilon
        p = 0.75
        q = 0.25

        sample = np.random.random()

        if bit == 1:
            if sample <= p:
                return 1
            else: return 0
        else:
            if sample == q:
                return 1
            else:
                return 0
    return [perturb_bit(b) for b in encoded_response]

perturb(encode('Sales'))

[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]

In [ ]:
# As in randomized response, flipping of bits causes issues in the answers
# This is the perturbed answer (without decoding)
counts = np.sum([perturb(encode(r)) for r in adult['Occupation']], axis=0)
list(zip(domain, counts))

In [ ]:
# This is the real answer
counts = np.sum([encode(r) for r in adult['Occupation']], axis=0)
list(zip(domain, counts))

## Question 5

Implement the *aggregate* step for Optimized Unary Hashing.

In [70]:
def aggregate(responses):
    p = 0.75
    q = 1 - p
    sums = np.sum(responses, axis=0)
    n = len(responses)
    return [(v-n*q) / (p-q) for v in sums]

responses = [perturb(encode(r)) for r in adult['Occupation']]
counts = aggregate(responses)
list(zip(domain, counts))

[('Adm-clerical', np.float64(-10578.5)),
 ('Exec-managerial', np.float64(-10178.5)),
 ('Handlers-cleaners', np.float64(-14258.5)),
 ('Prof-specialty', np.float64(-10034.5)),
 ('Other-service', np.float64(-11372.5)),
 ('Sales', np.float64(-10776.5)),
 ('Craft-repair', np.float64(-10162.5)),
 ('Transport-moving', np.float64(-13890.5)),
 ('Farming-fishing', np.float64(-14812.5)),
 ('Machine-op-inspct', np.float64(-13298.5)),
 ('Tech-support', np.float64(-14876.5)),
 ('Protective-serv', np.float64(-15328.5)),
 ('Armed-Forces', np.float64(-16264.5)),
 ('Priv-house-serv', np.float64(-16062.5))]